In [ ]:
!pip install facenet-pytorch scikit-learn tqdm pillow streamlit opencv-python-headless pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
real_csv_path = '/content/drive/MyDrive/faceforensics/csv/original.csv'
fake_csv_path = '/content/drive/MyDrive/faceforensics/csv/Deepfakes.csv'
base_video_dir = '/content/drive/MyDrive/faceforensics/'
real_df = pd.read_csv(real_csv_path)
fake_df = pd.read_csv(fake_csv_path)

In [ ]:
import cv2
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=False, post_process=True, device=device)

def extract_faces_from_csv(df, base_dir, output_dir, max_videos=50, max_frames=10):
    os.makedirs(output_dir, exist_ok=True)
    processed_count = 0

    # Loop over video rows listed in your CSV file
    for index, row in tqdm(df.iterrows(), total=min(len(df), max_videos)):
        if processed_count >= max_videos:
            break

        video_rel_path = row['File Path'] # e.g., 'original/000.mp4'
        full_video_path = os.path.join(base_dir, video_rel_path)

        if not os.path.exists(full_video_path):
            continue

        cap = cv2.VideoCapture(full_video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total_frames // max_frames)

        frame_idx = 0
        saved_frames = 0

        while cap.isOpened() and saved_frames < max_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb_frame)
            boxes, _ = mtcnn.detect(pil_img)

            if boxes is not None:
                box = boxes[0].astype(int)
                box[0], box[1] = max(0, box[0]), max(0, box[1])
                cropped_face = pil_img.crop((box[0], box[1], box[2], box[3]))

                video_name = os.path.basename(video_rel_path).split('.')[0]
                cropped_face.save(os.path.join(output_dir, f"{video_name}_f{frame_idx}.jpg"))
                saved_frames += 1

            frame_idx += step
        cap.release()
        processed_count += 1
extract_faces_from_csv(real_df, base_video_dir, "dataset/real", max_videos=30)
extract_faces_from_csv(fake_df, base_video_dir, "dataset/fake", max_videos=30)


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
100%|██████████| 30/30 [02:20<00:00,  4.67s/it]


In [ ]:
import os
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

gc.collect()
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 16
EPOCHS = 8
LEARNING_RATE = 2e-5

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root='dataset', transform=train_transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform

targets = [s[1] for s in full_dataset.samples]
class_counts = [targets.count(0), targets.count(1)]

class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[t] for t in targets]

train_indices = train_dataset.indices
train_sample_weights = [sample_weights[i] for i in train_indices]
sampler = WeightedRandomSampler(weights=train_sample_weights, num_samples=len(train_sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
model.heads.head = nn.Linear(model.heads.head.in_features, 2)

for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if scaler and device.type == 'cuda':
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            if device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    outputs = model(images)
            else:
                outputs = model(images)

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_acc = accuracy_score(all_labels, all_preds)

    if epoch_acc > best_acc:
        best_acc = epoch_acc
        os.makedirs("weights", exist_ok=True)
        checkpoint = {
            "state_dict": model.state_dict(),
            "class_to_idx": full_dataset.class_to_idx
        }
        torch.save(checkpoint, "weights/model_complete.pth")

 Training Device Active: cuda
 Dataset Mapping Detected: {'fake': 0, 'real': 1}
 Dataset Balance -> Class 0: 300 | Class 1: 300


Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:02<00:00, 131MB/s]
Epoch [1/8]: 100%|██████████| 30/30 [00:07<00:00,  4.21it/s, loss=0.495]


 Validation Accuracy for Epoch 1: 75.83%


Epoch [2/8]: 100%|██████████| 30/30 [00:06<00:00,  4.41it/s, loss=0.162]


 Validation Accuracy for Epoch 2: 91.67%


...

 Validation Accuracy for Epoch 6: 97.50%


Epoch [7/8]: 100%|██████████| 30/30 [00:06<00:00,  4.29it/s, loss=0.00262]


 Validation Accuracy for Epoch 7: 97.50%


Epoch [8/8]: 100%|██████████| 30/30 [00:07<00:00,  4.25it/s, loss=0.00254]


 Validation Accuracy for Epoch 8: 97.50%
 Final Best Accuracy: 98.33%
 Precision Score:    0.9751
 Recall Score:       0.9750
 F1 Score:          0.9750


In [ ]:
with open("app.py", "w") as f:
    f.write('''import streamlit as st
import torch
import torch.nn as nn
import cv2
import os
import json
import hashlib
import tempfile
import numpy as np
from PIL import Image
from torchvision import models, transforms, datasets
from facenet_pytorch import MTCNN

# --- PAGE CONFIGURATION ---
st.set_page_config(
    page_title="AegisFace - Deepfake Forensics Studio",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# --- CUSTOM CSS ---
st.markdown("""
    <style>
    .main { background-color: #0E1117; }
    .stMetric { background-color: #1E222D; padding: 15px; border-radius: 10px; border: 1px solid #2E3440; }
    .status-real { background-color: #1b4332; color: #2ecc71; padding: 14px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px; }
    .status-fake { background-color: #4a0e17; color: #e74c3c; padding: 14px; border-radius: 8px; font-weight: bold; text-align: center; font-size: 18px; }
    .stButton>button { width: 100%; border-radius: 8px; height: 42px; background-color: #4F46E5; color: white; font-weight: 600; }
    </style>
""", unsafe_allow_html=True)

USER_DB = "users.json"

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def load_users():
    if not os.path.exists(USER_DB):
        with open(USER_DB, "w") as f:
            json.dump({"admin": hash_password("admin123")}, f)
    with open(USER_DB, "r") as f:
        return json.load(f)

def save_user(username, password):
    users = load_users()
    users[username] = hash_password(password)
    with open(USER_DB, "w") as f:
        json.dump(users, f)

if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "username" not in st.session_state:
    st.session_state.username = ""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

@st.cache_resource
def init_detector():
    mtcnn = MTCNN(keep_all=False, post_process=False, select_largest=True, device=device)
    model = models.vit_b_16()
    model.heads.head = nn.Linear(model.heads.head.in_features, 2)
    if os.path.exists("weights/vit_deepfake.pth"):
        model.load_state_dict(torch.load("weights/vit_deepfake.pth", map_location=device))
    model.to(device).eval()

    # Auto-detect class mapping from local dataset
    real_idx = 1
    if os.path.exists("dataset"):
        try:
            ds = datasets.ImageFolder(root="dataset")
            real_idx = ds.class_to_idx.get("real", ds.class_to_idx.get("Real", 1))
        except Exception:
            real_idx = 1

    return mtcnn, model, real_idx

mtcnn, model, REAL_IDX = init_detector()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if not st.session_state.authenticated:
    st.title("🛡️ AegisFace Platform")
    st.caption("AI-Powered Deepfake Detection Suite")

    tab1, tab2 = st.tabs(["🔐 Sign In", "📝 Register"])
    with tab1:
        u = st.text_input("Username", key="login_user")
        p = st.text_input("Password", type="password", key="login_pwd")
        if st.button("Log In"):
            users = load_users()
            if u in users and users[u] == hash_password(p):
                st.session_state.authenticated = True
                st.session_state.username = u
                st.rerun()
            else:
                st.error("Invalid credentials.")
    with tab2:
        nu = st.text_input("New Username", key="reg_user")
        npw = st.text_input("New Password", type="password", key="reg_pwd")
        cpw = st.text_input("Confirm Password", type="password", key="reg_pwd_conf")
        if st.button("Register"):
            users = load_users()
            if not nu or not npw or npw != cpw or nu in users:
                st.error("Registration failed. Check password match or existing user.")
            else:
                save_user(nu, npw)
                st.success("Account created! Log in above.")
    st.stop()

with st.sidebar:
    st.title("🛡️ AegisFace AI")
    st.caption(f"Operator: {st.session_state.username}")
    st.markdown("---")
    st.text(f"Device: {device.type.upper()}")
    st.text(f"Model: ViT-B/16 Engine")
    st.text(f"Target Real Index: Class {REAL_IDX}")
    st.markdown("---")
    if st.button("Logout"):
        st.session_state.authenticated = False
        st.session_state.username = ""
        st.rerun()

st.title(" Video & Image Deepfake Forensics")
st.markdown("Upload a video (`.mp4`) or image (`.jpg`, `.png`) to perform facial spatial verification.")

uploaded_file = st.file_uploader("Drop Media File", type=["jpg", "png", "jpeg", "mp4"])

def extract_and_predict(pil_images):
    tensors = []
    crops = []

    for img in pil_images:
        # Detect boxes safely
        boxes, _ = mtcnn.detect(img)
        if boxes is not None and len(boxes) > 0:
            box = boxes[0].astype(int)
            w, h = img.size
            # Ensure valid box boundaries within image dimensions
            x1, y1 = max(0, box[0]), max(0, box[1])
            x2, y2 = min(w, box[2]), min(h, box[3])

            if x2 > x1 and y2 > y1:
                crop = img.crop((x1, y1, x2, y2))
                tensors.append(transform(crop))
                crops.append(crop)

    if not tensors:
        return None, []

    batch = torch.stack(tensors).to(device)
    with torch.no_grad():
        if device.type == 'cuda':
            with torch.cuda.amp.autocast():
                logits = model(batch)
        else:
            logits = model(batch)

        probs = torch.nn.functional.softmax(logits, dim=1)[:, REAL_IDX].cpu().numpy()

    return probs, crops

if uploaded_file:
    col1, col2 = st.columns([1, 1])

    # --- SINGLE IMAGE SCAN ---
    if uploaded_file.name.endswith(('.jpg', '.jpeg', '.png')):
        img = Image.open(uploaded_file).convert("RGB")
        with col1:
            st.image(img, caption="Target Input Image", use_container_width=True)
        with col2:
            st.subheader("Image Analysis Platform")
            if st.button("🔍 Submit Image for Scan"):
                with st.spinner("Processing face crop..."):
                    probs, crops = extract_and_predict([img])

                if probs is not None and len(probs) > 0:
                    real_prob = probs[0]
                    is_real = real_prob > 0.5
                    conf = real_prob if is_real else (1 - real_prob)

                    st.image(crops[0], caption="Extracted Face Box", width=180)

                    m1, m2 = st.columns(2)
                    m1.metric("Assessment", "REAL" if is_real else "DEEPFAKE")
                    m2.metric("Confidence", f"{conf * 100:.1f}%")

                    if is_real:
                        st.markdown('<div class="status-real">✅ VERDICT: REAL (Authentic)</div>', unsafe_allow_html=True)
                    else:
                        st.markdown('<div class="status-fake">🚨 VERDICT: DEEPFAKE DETECTED</div>', unsafe_allow_html=True)
                else:
                    st.warning("No faces detected in the provided image.")

    # --- VIDEO SCAN (WITH FIXED OpenCV RGB COLOR CORRECTION) ---
    elif uploaded_file.name.endswith('.mp4'):
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as tfile:
            tfile.write(uploaded_file.read())
            vid_path = tfile.name

        with col1:
            st.video(vid_path)

        with col2:
            st.subheader("Video Analysis Platform")
            if st.button("🚀 Submit Video for Scan"):
                with st.spinner("Decoding video frames with OpenCV..."):
                    cap = cv2.VideoCapture(vid_path)
                    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    sample_count = 24
                    step = max(1, total_frames // sample_count)

                    pil_frames = []
                    frame_idx = 0

                    while cap.isOpened() and len(pil_frames) < sample_count:
                        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                        ret, frame = cap.read()
                        if not ret or frame is None:
                            break

                        # CRITICAL FIX: Convert OpenCV BGR format to RGB properly
                        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        pil_frames.append(Image.fromarray(rgb_frame))
                        frame_idx += step

                    cap.release()

                    probs, crops = extract_and_predict(pil_frames)

                if probs is not None and len(probs) > 0:
                    avg_real_prob = np.mean(probs)
                    is_real = avg_real_prob > 0.5
                    conf = avg_real_prob if is_real else (1 - avg_real_prob)

                    st.markdown("**Sampled Facial Crops:**")
                    st.image(crops[:6], width=80)

                    m1, m2 = st.columns(2)
                    m1.metric("Video Assessment", "REAL" if is_real else "DEEPFAKE")
                    m2.metric("Average Confidence", f"{conf * 100:.1f}%")

                    if is_real:
                        st.markdown(f'<div class="status-real">✅ VERDICT: REAL VIDEO ({len(probs)} Frames Validated)</div>', unsafe_allow_html=True)
                    else:
                        st.markdown(f'<div class="status-fake">🚨 VERDICT: DEEPFAKE DETECTED ({len(probs)} Frames Validated)</div>', unsafe_allow_html=True)
                else:
                    st.warning("Could not isolate face bounding boxes from sampled frames.")
''')

In [ ]:
import os
import subprocess
import time

# 1. Download and install cloudflared directly in Colab if not present
if not os.path.exists('/usr/local/bin/cloudflared'):
    print(" Installing Cloudflare Tunnel CLI...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("Cloudflare installed successfully!")

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)

print(" LAUNCHING CLOUDFLARE TUNNEL...")
!cloudflared tunnel --url http://localhost:8501 2>&1 | grep -o 'https://.*\.trycloudflare\.com'

 Installing Cloudflare Tunnel CLI...
Cloudflare installed successfully!
 LAUNCHING CLOUDFLARE TUNNEL...
https://purse-convertible-frequency-massachusetts.trycloudflare.com
https://purse-convertible-frequency-massachusetts.trycloudflare.com
https://purse-convertible-frequency-massachusetts.trycloudflare.com
